# Advanced Python descriptors and polygon design — a worked-problem tutorial

**A completely new exercise set with detailed solutions, small experiments, and many explanation cells.**

This notebook takes the same subject and teaching rhythm as the supplied notebook: first build an integer descriptor, then a point class, then a descriptor for a sequence of points, then polygons and subclasses. Each advanced problem is developed in a series of logical steps: a prediction, one concept, a short code example, a test, and an explanation of the result. The examples here are new; they are **not** a reprint or reformatting of the previous problem set.

**Level:** advanced Python / object model / careful API design. **Requirements:** Python 3.9+ and only the standard library. Run the cells **in order** in a fresh kernel. Deliberate failures are caught and tested, so “Run All” should finish successfully.

**Learning goals:** understand `__set_name__`, strict integer validation, `__slots__`, defensive snapshots, the sequence ABC, atomic updates, subclass-specific descriptor constraints, geometric invariants, class-creation hooks, robust point classification, and invariant-based testing.

## How to use the notebook

1. Read the **problem** and **checkpoint** before running the solution.
2. Execute one small code cell at a time. The accompanying prose explains what each result proves—and what it does *not* prove.
3. Treat an `assert` as an executable claim about your implementation. For invalid inputs we test the expected exception *and* confirm that existing state is unchanged.
4. The coordinate system is a screen-like rectangle: `0 <= x <= 800` and `0 <= y <= 400`. Coordinates must be **actual `int` objects**; `bool` is rejected even though it subclasses `int`.
5. A polygon is represented by vertices in **boundary order**. It is not automatically a simple, convex, square, or even nondegenerate polygon; those requirements are added only when a problem explicitly introduces them.

### Route through the tutorial

- Problems 1–4: descriptor naming, coordinate slots, defensive vertex storage, sequence interface.
- Problems 5–6: all-or-nothing editing and count-specific subclasses.
- Problems 7–8: simple and strictly convex geometry as validation hooks.
- Problems 9–10: dynamic class construction and winding direction.
- Problems 11–12: classify points and perform safe geometric transformations.
- Problems 13–14: descriptor storage traps, stress tests, and an integrated capstone.

## A tiny test helper

Invalid-input examples are valuable, but an unhandled exception would stop a notebook's “Run All”. This helper checks exactly the expected exception class and optionally a useful fragment of its message. Any different failure still propagates, revealing a real bug.

In [1]:
from collections.abc import Sequence
from operator import setitem
import random
import weakref


def expect_error(error_type, operation, message_part=None):
    """Assert that zero-argument operation raises exactly error_type or a subclass."""
    try:
        operation()
    except error_type as exc:
        if message_part is not None:
            assert message_part in str(exc), (message_part, str(exc))
        return str(exc)
    raise AssertionError(f"Expected {error_type.__name__} but nothing was raised")


print('Test helper ready')

Test helper ready


## Problem 01 — Why reusing ONE descriptor for TWO names is a trap

**The problem.** You want `x` and `y` to have identical behavior and decide to write `x = y = OneDescriptor()`. The code looks economical. Is it correct?

**Your task.** Predict the calls to `__set_name__`, reproduce the bug, and fix it without changing the public attribute names.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Track class creation rather than guessing

`__set_name__(owner, name)` is invoked when the owner class is created. A descriptor instance can be bound to more than one name, but a descriptor that stores a *single* `self.name` will overwrite its own earlier binding. We first make that mechanism visible.

In [2]:
class NamedSlot:
    def __set_name__(self, owner, name):
        print(f"binding {owner.__name__}.{name}")
        self.storage_name = '_' + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)


shared_descriptor = NamedSlot()

class BrokenPair:
    x = shared_descriptor
    y = shared_descriptor


pair = BrokenPair()
pair.x = 10
pair.y = 20
print('x =', pair.x, ', y =', pair.y)
assert pair.x == pair.y == 20

binding BrokenPair.x
binding BrokenPair.y
x = 20 , y = 20


**What happened?** The class body bound the *same object* twice. The second `__set_name__` changed its storage key to `_y`; both attribute names now read and write that one key. This is a naming bug, not a flaw in the descriptor protocol.

**Design rule:** allocate a separate stateful descriptor instance for each named attribute. Sharing descriptor *classes* is excellent; sharing an instance that holds a single name is not.

### Step 2 — Fix it with two instances and assert storage separation

Keep the reusable `NamedSlot` implementation, but allocate a descriptor once per declaration. This is also how we will declare coordinate constraints later.

In [3]:
class CorrectPair:
    x = NamedSlot()
    y = NamedSlot()


pair = CorrectPair()
pair.x = 10
pair.y = 20
assert (pair.x, pair.y) == (10, 20)
assert pair.__dict__ == {'_x': 10, '_y': 20}
assert CorrectPair.x is not CorrectPair.y
print(pair.__dict__)

binding CorrectPair.x
binding CorrectPair.y
{'_x': 10, '_y': 20}


**Checkpoint.** Does `CorrectPair.x` return a point coordinate?

**Solution / reasoning.** No. Access through the class invokes `__get__(None, CorrectPair)`, so our descriptor returns itself. Class access makes metadata and configuration inspectable.

## Problem 02 — Build strict bounded coordinates without an instance `__dict__`

**The problem.** The source material stores descriptor-managed values in `instance.__dict__`. Can a descriptor also manage an object that uses `__slots__`?

**Your task.** Implement a reusable integer descriptor with strict type checks, an inclusive range, unset-attribute behavior, class-level introspection, and a no-deletion rule.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Select an appropriate storage strategy

Each coordinate descriptor will use a dedicated private slot, `_x` or `_y`. `__set_name__` derives the private name. `getattr` without a default raises `AttributeError` if an instance has not yet been initialized; unlike silently returning `None`, that makes the missing state explicit. The class must declare the slots that its descriptors intend to use.

In [4]:
class StrictBoundedInt:
    """A data descriptor whose value lives in an instance's private slot."""

    def __init__(self, minimum=None, maximum=None):
        for name, bound in (('minimum', minimum), ('maximum', maximum)):
            if bound is not None and type(bound) is not int:
                raise TypeError(f'{name} must be an exact int or None')
        if minimum is not None and maximum is not None and minimum > maximum:
            raise ValueError('minimum cannot exceed maximum')
        self.minimum, self.maximum = minimum, maximum

    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = '_' + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.public_name} must be an exact int')
        if self.minimum is not None and value < self.minimum:
            raise ValueError(f'{self.public_name} must be >= {self.minimum}')
        if self.maximum is not None and value > self.maximum:
            raise ValueError(f'{self.public_name} must be <= {self.maximum}')
        setattr(instance, self.storage_name, value)

    def __delete__(self, instance):
        raise AttributeError(f'{self.public_name} cannot be deleted')

**Why `type(value) is int`?** In Python, `isinstance(True, int)` is true. Screen coordinates often should not accept booleans, and a subclass of `int` may have specialized behavior. Here, “strict” is an explicit contract—not a universal rule for every descriptor. `TypeError` describes a wrong kind of value; `ValueError` describes an integer outside the permitted range.

### Step 2 — Construct a mutable point with value equality

We deliberately make points **mutable** in this tutorial. That choice creates an aliasing problem that Problem 3 must solve. We implement value-based equality but do **not** implement hashing: a mutable, value-equal object must not be used as a normal dictionary key.

In [5]:
class Point2D:
    __slots__ = ('_x', '_y')
    x = StrictBoundedInt(minimum=0, maximum=800)
    y = StrictBoundedInt(minimum=0, maximum=400)

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point2D({self.x}, {self.y})'

    def __str__(self):
        return f'({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Point2D):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)


p = Point2D(12, 34)
assert repr(p) == 'Point2D(12, 34)'
assert str(p) == '(12, 34)'
assert Point2D(12, 34) == p
assert not hasattr(p, '__dict__')
assert Point2D.x.minimum == 0
print(p, '| class descriptor:', Point2D.y.maximum)

(12, 34) | class descriptor: 400


### Step 3 — Probe the bad inputs individually

A clean test should distinguish a bad *type*, an out-of-range *value*, and an illegal *deletion*. A failed setter must leave the old value intact because validation happens before storage.

In [6]:
assert 'exact int' in expect_error(TypeError, lambda: Point2D(True, 4))
assert 'exact int' in expect_error(TypeError, lambda: Point2D(2.0, 4))
assert '<= 800' in expect_error(ValueError, lambda: Point2D(801, 4))
assert '>= 0' in expect_error(ValueError, lambda: Point2D(1, -1))

p = Point2D(1, 2)
expect_error(ValueError, lambda: setattr(p, 'x', 999))
assert p.x == 1
expect_error(AttributeError, lambda: delattr(p, 'x'))
assert p.x == 1
expect_error(TypeError, lambda: hash(p))
print('Wrong types, bounds, deletion, rollback, and mutable hashing: all checked')

Wrong types, bounds, deletion, rollback, and mutable hashing: all checked


### Step 4 — Test uninitialized points explicitly

`__new__` creates a point without running `__init__`. That is not normal client code, but it is a useful protocol test: our descriptor should report the missing private slot instead of manufacturing a default.

In [7]:
uninitialized = Point2D.__new__(Point2D)
assert expect_error(AttributeError, lambda: uninitialized.x)
assert not hasattr(uninitialized, '_x')
print('Unset coordinate correctly raises AttributeError')

Unset coordinate correctly raises AttributeError


**Checkpoint.** Why do we not implement `__hash__ = hash((self.x, self.y))`?

**Solution / reasoning.** Because users can later change `x` or `y`. Its hash would then change, violating the expectations of sets and dictionaries. With `__eq__` but no `__hash__`, Python makes this class unhashable, which is appropriate for our mutable points.

## Problem 03 — A vertex descriptor that defeats accidental aliasing

**The problem.** A polygon receives a list of mutable `Point2D` objects. The caller changes that list or changes an original point after construction. Should the polygon silently change?

**Your task.** Accept only a sequence of correctly typed points; check length; store private immutable coordinate pairs; and return independent point snapshots on every read.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Separate the external API from the internal representation

The public value is a tuple of `Point2D` objects. But a tuple only prevents *replacing tuple entries*—it does not freeze the mutable points inside. Instead, we will store a tuple of **integer pairs** internally, and reconstruct fresh points on reads. The descriptor also calls an overridable validation hook before committing a candidate. That hook will become important for geometric subclasses.

In [8]:
class VertexSequence:
    """Validate point sequences; privately store immutable (x, y) pairs."""

    def __init__(self, min_length=3, max_length=None):
        if type(min_length) is not int or min_length < 0:
            raise ValueError('min_length must be a nonnegative int')
        if max_length is not None:
            if type(max_length) is not int or max_length < min_length:
                raise ValueError('max_length must be >= min_length')
        self.min_length, self.max_length = min_length, max_length

    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = '_' + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        stored = getattr(instance, self.storage_name)
        return tuple(Point2D(x, y) for x, y in stored)

    def __set__(self, instance, value):
        if not isinstance(value, Sequence):
            raise TypeError(f'{self.public_name} must be a sequence')
        if len(value) < self.min_length:
            raise ValueError(f'{self.public_name} needs at least {self.min_length} points')
        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(f'{self.public_name} allows at most {self.max_length} points')

        candidate = []
        for position, point in enumerate(value):
            if not isinstance(point, Point2D):
                raise TypeError(f'{self.public_name}[{position}] must be a Point2D')
            candidate.append((point.x, point.y))
        candidate = tuple(candidate)
        instance._validate_vertices(candidate)  # Verify BEFORE committing.
        setattr(instance, self.storage_name, candidate)

    def __delete__(self, instance):
        raise AttributeError(f'{self.public_name} cannot be deleted')

**Invariants built into the setter:** length, element type, and an immutable stored snapshot. Most importantly, the only write to the private slot occurs *after* all checks pass. This supplies a simple all-or-nothing guarantee for regular descriptor assignments; it does not make an object safe against deliberate private-slot tampering or simultaneous edits from multiple threads.

### Step 2 — Make the smallest host class

For now, the hook does nothing. We use a small host rather than introducing the full polygon interface too early: each new concept should be testable in isolation.

In [9]:
class VertexSketch:
    __slots__ = ('_vertices',)
    vertices = VertexSequence(min_length=3, max_length=6)

    def __init__(self, vertices):
        self.vertices = vertices

    def _validate_vertices(self, candidate):
        """Subclasses can impose stronger requirements on immutable pairs."""
        return None


incoming = [Point2D(0, 0), Point2D(4, 0), Point2D(0, 4)]
sketch = VertexSketch(incoming)
assert len(sketch.vertices) == 3
print('Public snapshots:', sketch.vertices)
print('Private immutable pairs:', sketch._vertices)

Public snapshots: (Point2D(0, 0), Point2D(4, 0), Point2D(0, 4))
Private immutable pairs: ((0, 0), (4, 0), (0, 4))


### Step 3 — Attack all three aliasing routes

The caller might mutate (a) the original list, (b) an original point, or (c) a point returned by the public getter. These are different paths. Check each one explicitly.

In [10]:
incoming.append(Point2D(8, 8))
incoming[0].x = 777
returned = sketch.vertices
returned[1].y = 333
assert sketch._vertices == ((0, 0), (4, 0), (0, 4))
assert len(sketch.vertices) == 3
assert sketch.vertices[0] == Point2D(0, 0)
assert sketch.vertices[1] == Point2D(4, 0)
print('Original list, original point, and returned point cannot mutate storage')

Original list, original point, and returned point cannot mutate storage


### Step 4 — Test rejection and rollback

Compare a saved immutable snapshot before and after invalid input. We are testing the resulting **state**, not just the exception message.

In [11]:
before = sketch._vertices
expect_error(TypeError, lambda: setattr(sketch, 'vertices', (p for p in incoming)), 'sequence')
expect_error(ValueError, lambda: setattr(sketch, 'vertices', [Point2D(1, 1)]), 'at least')
expect_error(TypeError, lambda: setattr(sketch, 'vertices', [Point2D(0, 0), 'oops', Point2D(1, 1)]), '[1]')
expect_error(ValueError, lambda: setattr(sketch, 'vertices', [Point2D(1, 1)] * 7), 'at most')
assert sketch._vertices == before
assert VertexSketch.vertices.max_length == 6
print('All rejection paths preserved the prior value')

All rejection paths preserved the prior value


**Checkpoint.** Why not silently convert a generator into a list in `VertexSequence.__set__`?

**Solution / reasoning.** The stated API requires a `Sequence`, which has a length and supports indexed access. Accepting arbitrary iterators would change that contract and raise additional questions about unbounded streams and partial consumption. An explicit `extend` operation can materialize an iterable when that behavior is desired.

## Problem 04 — Make the polygon a real read-only sequence of points

**The problem.** Clients want indexing, slicing, iteration, `len`, and containment—but should not be able to mutate internal storage through those operations.

**Your task.** Implement the `collections.abc.Sequence` contract on top of the descriptor and explain the difference between a read-only *view* and a detached *snapshot*.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Implement the two essential sequence methods

The `Sequence` ABC supplies useful mixin methods if we implement `__len__` and `__getitem__`. We also give `__iter__` and `__contains__` explicit implementations so their snapshot behavior is obvious. A slice of our public tuple returns another tuple, never a writable alias of internal storage.

In [12]:
class Polygon(Sequence):
    __slots__ = ('_vertices',)
    vertices = VertexSequence(min_length=3)

    def __init__(self, *vertices):
        self.vertices = vertices

    def _validate_vertices(self, candidate):
        return None

    def __len__(self):
        return len(self._vertices)

    def __getitem__(self, key):
        return self.vertices[key]

    def __iter__(self):
        return iter(self.vertices)

    def __contains__(self, item):
        return item in self.vertices

    def __repr__(self):
        points = ', '.join(repr(p) for p in self.vertices)
        return f'{type(self).__name__}({points})'


poly = Polygon(Point2D(0, 0), Point2D(4, 0), Point2D(0, 4))
assert isinstance(poly, Sequence)
assert len(poly) == 3
assert poly[0] == Point2D(0, 0)
assert poly[-1] == Point2D(0, 4)
assert isinstance(poly[:2], tuple)
assert Point2D(4, 0) in poly
print('Sequence interface:', list(poly), '| slice:', poly[:2])

Sequence interface: [Point2D(0, 0), Point2D(4, 0), Point2D(0, 4)] | slice: (Point2D(0, 0), Point2D(4, 0))


### Step 2 — Check what a returned element can and cannot do

Our point class is mutable, and we do not pretend otherwise. The key is that `poly[0]` is a *fresh* point. Mutating that point affects only the detached copy.

In [13]:
first = poly[0]
first.x = 100
assert poly[0] == Point2D(0, 0)
expect_error(TypeError, lambda: setitem(poly.vertices, 0, Point2D(2, 2)))
expect_error(AttributeError, lambda: delattr(poly, 'vertices'))
print('The public sequence cannot change the stored tuple')

The public sequence cannot change the stored tuple


**Subtle distinction:** `Sequence` describes supported operations, not “deep immutability”. In our design, immutability comes from *private coordinate pairs* and a getter that makes fresh objects. No convenience method should expose `self._vertices` as a mutable list later on.

**Cost:** reading the entire `vertices` property takes O(n) time and creates n points. This is a deliberate safety/performance tradeoff, suitable for small-to-medium polygon data. Large numerical geometry libraries often use different representations and ownership contracts.

**Checkpoint.** Why is `list(poly)` safe when `Point2D` is mutable?

**Solution / reasoning.** `__iter__` iterates over a freshly materialized tuple of fresh points. The resulting list is writable, but modifying it or its points does not modify `poly._vertices`.

## Problem 05 — Design editing methods with a single commit point

**The problem.** A polygon needs append, extend, replacement, and deletion operations. A naive implementation modifies a list first and validates later, leaving half-finished state after an error.

**Your task.** Implement candidate → validate → commit for each operation, and test that a failed edit cannot corrupt the previous polygon.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Begin with append and extend

Build a new tuple of public snapshots, then assign it through the descriptor. `extend` accepts any **finite** iterable, unlike the descriptor setter. It materializes that iterable once, before the single commit. Materializing an infinite iterable would never terminate; callers must not pass one.

In [14]:
class EditablePolygon(Polygon):
    def append(self, point):
        self.vertices = (*self.vertices, point)

    def extend(self, points):
        additions = tuple(points)
        self.vertices = (*self.vertices, *additions)


edit = EditablePolygon(Point2D(0, 0), Point2D(4, 0), Point2D(0, 4))
edit.append(Point2D(2, 2))
edit.extend(iter([Point2D(3, 1), Point2D(3, 2)]))
assert len(edit) == 6
print('After two operations:', edit)

After two operations: EditablePolygon(Point2D(0, 0), Point2D(4, 0), Point2D(0, 4), Point2D(2, 2), Point2D(3, 1), Point2D(3, 2))


### Step 2 — Measure rollback under an invalid extension

A generator can produce valid items and then a bad item. Because nothing is assigned until the whole candidate validates, the edit must fail without retaining its valid prefix.

In [15]:
before = edit._vertices
bad_stream = (item for item in (Point2D(5, 5), 'invalid', Point2D(6, 6)))
expect_error(TypeError, lambda: edit.extend(bad_stream), '[7]')
assert edit._vertices == before
print('Invalid element in an extension: no partial append')

Invalid element in an extension: no partial append


### Step 3 — Add replacement and deletion as complete reassignments

Operations that remove items can violate `min_length`. Slicing a candidate list is safe because it is an independent copy. As before, the descriptor is the only commit gate.

In [16]:
class EditingPolygon(EditablePolygon):
    def replace_at(self, position, point):
        candidate = list(self.vertices)
        candidate[position] = point
        self.vertices = candidate

    def remove_at(self, position):
        candidate = list(self.vertices)
        removed = candidate.pop(position)
        self.vertices = candidate
        return removed


edit = EditingPolygon(Point2D(0, 0), Point2D(4, 0), Point2D(0, 4), Point2D(2, 2))
edit.replace_at(3, Point2D(3, 3))
removed = edit.remove_at(3)
assert removed == Point2D(3, 3)
assert len(edit) == 3
print('Editing and removal:', edit)

Editing and removal: EditingPolygon(Point2D(0, 0), Point2D(4, 0), Point2D(0, 4))


### Step 4 — Check the minimum-length boundary

Deletion of the third point would leave only two vertices. The candidate is rejected, and the original three-vertex polygon survives.

In [17]:
before = edit._vertices
expect_error(ValueError, lambda: edit.remove_at(0), 'at least')
assert edit._vertices == before
expect_error(IndexError, lambda: edit.replace_at(99, Point2D(1, 1)))
assert edit._vertices == before
print('Minimum length and invalid index preserved previous state')

Minimum length and invalid index preserved previous state


**Checkpoint.** Why did we not implement `poly.vertices.append(point)`?

**Solution / reasoning.** `vertices` returns a tuple and therefore exposes no `.append`. Requiring edits through explicit methods forces each operation to pass through the entire validation pipeline. A live mutable list would bypass those checks.

## Problem 06 — Specialize vertex counts without duplicating algorithms

**The problem.** A triangle must always contain exactly three points, while a quadrilateral needs exactly four. The edit methods already exist in a reusable base class.

**Your task.** Override only the descriptor configuration, demonstrate inherited edit behavior, and explain why “four points” is not enough to claim “square”.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Override the descriptor, not the editing methods

A subclass declaration creates its own `VertexSequence` instance. When inherited methods assign `self.vertices`, normal attribute lookup finds the subclass descriptor. Its own `max_length` and `min_length` are therefore enforced without copying any code.

In [18]:
class Triangle(EditingPolygon):
    vertices = VertexSequence(min_length=3, max_length=3)


class Quadrilateral(EditingPolygon):
    vertices = VertexSequence(min_length=4, max_length=4)


tri = Triangle(Point2D(0, 0), Point2D(3, 0), Point2D(0, 3))
quad = Quadrilateral(Point2D(0, 0), Point2D(4, 0), Point2D(4, 4), Point2D(0, 4))
assert type(tri).vertices.max_length == 3
assert type(quad).vertices.min_length == 4
assert Polygon.vertices.max_length is None
print('Independent constraints:', Triangle.vertices.max_length, Quadrilateral.vertices.max_length)

Independent constraints: 3 4


### Step 2 — Inherited algorithms acquire stricter behavior automatically

Try extending or deleting a triangle. Then replace one of its points: replacement keeps the count at three and is allowed. Importantly, count validation alone says nothing about shape.

In [19]:
tri_before = tri._vertices
expect_error(ValueError, lambda: tri.append(Point2D(2, 2)), 'at most')
expect_error(ValueError, lambda: tri.remove_at(0), 'at least')
assert tri._vertices == tri_before
tri.replace_at(2, Point2D(1, 2))
assert tri[2] == Point2D(1, 2)
print('A count-preserving edit succeeds; count-changing edits fail')

A count-preserving edit succeeds; count-changing edits fail


**Important limitation:** `Triangle` currently means “a sequence containing three validated points.” Three collinear points still pass. Likewise, `Quadrilateral` means “four points,” not “a square,” not necessarily “simple,” and not even necessarily “nondegenerate.” The next two problems introduce those geometric requirements explicitly, rather than silently claiming them here.

In [20]:
collinear = Triangle(Point2D(0, 0), Point2D(2, 2), Point2D(4, 4))
assert len(collinear) == 3
print('Count-only Triangle accepts collinear vertices:', collinear)

Count-only Triangle accepts collinear vertices: Triangle(Point2D(0, 0), Point2D(2, 2), Point2D(4, 4))


**Checkpoint.** Why does `Triangle.append` respect `Triangle.vertices` rather than the base class descriptor?

**Solution / reasoning.** The inherited method assigns through `self.vertices` on a Triangle instance. Descriptor lookup follows that instance’s class and finds the overriding `Triangle.vertices`.

## Problem 07 — Reject self-intersecting and degenerate polygons

**The problem.** Three or more valid points do not guarantee a valid geometric boundary. A polygon can repeat vertices, retrace an edge, self-intersect, or have zero signed area.

**Your task.** Write exact integer orientation and segment predicates. Use the existing pre-commit hook to enforce a simple, nonzero-area polygon, including on inherited editing methods.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Use exact cross products, not slopes

For integer points `a`, `b`, `c`, the 2D cross product `(b-a) × (c-a)` is an integer. Its sign tells whether `c` lies left of, right of, or on the directed segment `a → b`. Unlike slope division, this method handles vertical lines without special cases and avoids floating-point error.

In [21]:
def turn(a, b, c):
    """Signed twice-triangle-area for coordinate pairs a, b, c."""
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])


def point_on_segment(a, b, p):
    return (turn(a, b, p) == 0
            and min(a[0], b[0]) <= p[0] <= max(a[0], b[0])
            and min(a[1], b[1]) <= p[1] <= max(a[1], b[1]))


assert turn((0, 0), (4, 0), (2, 1)) > 0
assert turn((0, 0), (4, 0), (2, -1)) < 0
assert point_on_segment((0, 0), (4, 0), (2, 0))
assert not point_on_segment((0, 0), (4, 0), (5, 0))
print('Orientation and closed-segment predicates verified')

Orientation and closed-segment predicates verified


### Step 2 — Implement a complete segment intersection test

A pair of segments can intersect strictly in the interior or touch at an endpoint. Collinear overlaps matter too. First test collinear endpoints with `point_on_segment`; otherwise, two pairs of orientation signs must differ.

In [22]:
def segments_intersect(a, b, c, d):
    t1, t2 = turn(a, b, c), turn(a, b, d)
    t3, t4 = turn(c, d, a), turn(c, d, b)
    if t1 == 0 and point_on_segment(a, b, c):
        return True
    if t2 == 0 and point_on_segment(a, b, d):
        return True
    if t3 == 0 and point_on_segment(c, d, a):
        return True
    if t4 == 0 and point_on_segment(c, d, b):
        return True
    return (t1 > 0) != (t2 > 0) and (t3 > 0) != (t4 > 0)


assert segments_intersect((0, 0), (4, 4), (0, 4), (4, 0))
assert segments_intersect((0, 0), (4, 0), (2, 0), (6, 0))
assert not segments_intersect((0, 0), (1, 0), (2, 0), (3, 0))
print('Crossing, collinear overlap, and disjoint cases checked')

Crossing, collinear overlap, and disjoint cases checked


### Step 3 — Recognize which edges are adjacent

For n vertices, edge i runs from vertex i to vertex `(i+1) % n`. Edge i and edge i+1 share an endpoint by design, as do edges 0 and n−1. We should not flag these intended contacts as self-intersections. Instead, test all **nonadjacent** edge pairs. We also reject a collinear backtrack at consecutive vertices.

In [23]:
def signed_twice_area(coords):
    return sum(coords[i][0] * coords[(i + 1) % len(coords)][1]
               - coords[(i + 1) % len(coords)][0] * coords[i][1]
               for i in range(len(coords)))


def validate_simple_boundary(coords):
    n = len(coords)
    if len(set(coords)) != n:
        raise ValueError('vertices must be distinct')

    for i in range(n):
        a, b, c = coords[i - 1], coords[i], coords[(i + 1) % n]
        dot = ((a[0] - b[0]) * (c[0] - b[0])
               + (a[1] - b[1]) * (c[1] - b[1]))
        if turn(a, b, c) == 0 and dot > 0:
            raise ValueError('boundary must not backtrack along an edge')

    for i in range(n):
        a, b = coords[i], coords[(i + 1) % n]
        for j in range(i + 1, n):
            if j == i + 1 or (i == 0 and j == n - 1):
                continue  # Adjacent edges share a legitimate endpoint.
            c, d = coords[j], coords[(j + 1) % n]
            if segments_intersect(a, b, c, d):
                raise ValueError(f'nonadjacent edges {i} and {j} intersect')

    if signed_twice_area(coords) == 0:
        raise ValueError('polygon area must not be zero')

**Definition used here:** a *simple polygon* has a non-self-intersecting closed boundary; we additionally require distinct vertices and nonzero area. Consecutive collinear vertices are allowed when they travel forward along an edge, but an immediate backtrack is rejected. These choices are explicit; some geometric APIs make different choices about collinear subdivisions.

**Complexity:** the intersection loop examines O(n²) pairs. That is clear and reliable for an educational polygon with modest vertex counts. Production geometry tools may use sweep-line algorithms for very large inputs.

### Step 4 — Attach the validator with no change to the descriptor

The `VertexSequence` setter already calls `instance._validate_vertices(candidate)` before writing `_vertices`. A subclass merely overrides the hook. That means constructor assignments and later descriptor assignments both enforce the stronger condition.

In [24]:
class SimplePolygon(Polygon):
    def _validate_vertices(self, candidate):
        validate_simple_boundary(candidate)


square = SimplePolygon(Point2D(0, 0), Point2D(4, 0),
                       Point2D(4, 4), Point2D(0, 4))
assert signed_twice_area(square._vertices) == 32
print('Simple polygon accepted:', square)

Simple polygon accepted: SimplePolygon(Point2D(0, 0), Point2D(4, 0), Point2D(4, 4), Point2D(0, 4))


### Step 5 — Prove that invalid geometry really is rejected

Try a crossing bow tie, repeated vertex, and a collinear triple. Each has valid coordinate types and lengths. Their rejection therefore demonstrates the *geometric* layer rather than the descriptor’s elementary checks.

In [25]:
def simple_of(*pairs):
    return SimplePolygon(*(Point2D(*xy) for xy in pairs))

expect_error(ValueError, lambda: simple_of((0, 0), (4, 4), (0, 4), (4, 0)), 'intersect')
expect_error(ValueError, lambda: simple_of((0, 0), (4, 0), (4, 0), (0, 4)), 'distinct')
expect_error(ValueError, lambda: simple_of((0, 0), (2, 2), (4, 4)), 'backtrack')
print('Crossing, repeated vertex, and collinear/backtracking cases rejected')

Crossing, repeated vertex, and collinear/backtracking cases rejected


### Step 6 — Test an invalid edit against an existing simple polygon

Direct assignment is itself a transaction. We test it without yet mixing in editing methods. This is a strong example of **reuse**: the same descriptor has acquired a new validation rule through polymorphism.

In [26]:
old_square = square._vertices
bad = [Point2D(0, 0), Point2D(4, 4), Point2D(0, 4), Point2D(4, 0)]
expect_error(ValueError, lambda: setattr(square, 'vertices', bad), 'intersect')
assert square._vertices == old_square
print('SimplePolygon rejected invalid reassignment without damage')

SimplePolygon rejected invalid reassignment without damage


**Checkpoint.** Would checking `len(set(vertices)) == len(vertices)` by itself prevent all self-intersections?

**Solution / reasoning.** No. A bow tie can have four distinct corners but crossing nonadjacent edges. Distinctness, edge-intersection checks, and nonzero area are separate requirements.

## Problem 08 — Demand strict convexity without changing the setter

**The problem.** A simple polygon may have an inward indentation. Another application requires every consecutive corner to turn in a consistent nonzero direction.

**Your task.** Extend the simple-polygon hook to enforce strict convexity, and explain why a clockwise polygon should be treated identically to a counterclockwise one.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Compute all consecutive turn signs

A strictly convex polygon has no collinear consecutive triple, and all consecutive cross products have the same sign. Positive and negative signs correspond to opposite vertex orders; neither orientation is inherently invalid. We first delegate to `SimplePolygon` to guarantee the simpler boundary properties.

In [27]:
class StrictConvexPolygon(SimplePolygon):
    def _validate_vertices(self, candidate):
        super()._validate_vertices(candidate)
        signs = []
        for i in range(len(candidate)):
            cross = turn(candidate[i - 1], candidate[i],
                         candidate[(i + 1) % len(candidate)])
            if cross == 0:
                raise ValueError('strict convexity forbids collinear corners')
            signs.append(cross > 0)
        if len(set(signs)) != 1:
            raise ValueError('all corners must turn in the same direction')


convex = StrictConvexPolygon(Point2D(0, 0), Point2D(4, 0),
                             Point2D(4, 4), Point2D(0, 4))
reverse_convex = StrictConvexPolygon(*reversed(convex.vertices))
assert signed_twice_area(convex._vertices) == -signed_twice_area(reverse_convex._vertices)
print('Both orientations satisfy strict convexity')

Both orientations satisfy strict convexity


### Step 2 — Distinguish a dent from a redundant collinear corner

The following concave pentagon is simple but not convex. The following five-vertex boundary traces a square with an additional collinear point on its bottom edge; it is simple, but **not strictly** convex. Test those two notions independently.

In [28]:
concave_pairs = ((0, 0), (6, 0), (6, 6), (3, 3), (0, 6))
concave = simple_of(*concave_pairs)
assert len(concave) == 5
expect_error(ValueError,
             lambda: StrictConvexPolygon(*(Point2D(*p) for p in concave_pairs)),
             'same direction')
expect_error(ValueError,
             lambda: StrictConvexPolygon(Point2D(0, 0), Point2D(2, 0),
                                         Point2D(4, 0), Point2D(4, 4), Point2D(0, 4)),
             'collinear')
print('Concavity and collinear turns are rejected for different reasons')

Concavity and collinear turns are rejected for different reasons


### Step 3 — Combine two independent constraints in a three-sided subclass

A three-sided strictly convex polygon needs both exact length and valid geometry. Override the descriptor for count; inherit the geometric hook. This is composition of contracts through normal class inheritance.

In [29]:
class GeometricTriangle(StrictConvexPolygon):
    vertices = VertexSequence(min_length=3, max_length=3)


geo_tri = GeometricTriangle(Point2D(0, 0), Point2D(3, 0), Point2D(0, 3))
expect_error(ValueError, lambda: setattr(geo_tri, 'vertices',
                                         (Point2D(0, 0), Point2D(1, 1), Point2D(2, 2))),
             'backtrack')
expect_error(ValueError, lambda: setattr(geo_tri, 'vertices', (*geo_tri.vertices, Point2D(1, 1))),
             'at most')
assert len(geo_tri) == 3
print('Geometric triangle enforces both geometry and exact vertex count')

Geometric triangle enforces both geometry and exact vertex count


**Checkpoint.** Why is it wrong to demand “all turns > 0”?

**Solution / reasoning.** That would reject an equally valid convex polygon whose vertices are listed clockwise. Strict convexity is about *consistent nonzero signs*, not choosing a particular winding direction.

## Problem 09 — Generate fixed-size subclasses using `__init_subclass__`

**The problem.** You need dozens of polygon types with counts 3, 4, 5, and so on. Repeating an entire subclass body for each count is unnecessary. Can class creation generate a descriptor automatically?

**Your task.** Implement a declarative `vertex_count` class keyword; account for the important fact that adding a descriptor *after* class creation does not automatically invoke `__set_name__`.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — State the class-creation wrinkle

Descriptors declared inside a class body receive `__set_name__` automatically. In contrast, a descriptor assigned to an existing class **later** does not. `__init_subclass__` runs after the new subclass object has been created, so we must call `__set_name__` ourselves for the descriptor we install there.

In [30]:
class FixedSizePolygon(EditingPolygon):
    def __init_subclass__(cls, *, vertex_count, **kwargs):
        super().__init_subclass__(**kwargs)
        if type(vertex_count) is not int or vertex_count < 3:
            raise ValueError('vertex_count must be an exact integer >= 3')
        cls.vertex_count = vertex_count
        descriptor = VertexSequence(min_length=vertex_count,
                                    max_length=vertex_count)
        cls.vertices = descriptor
        descriptor.__set_name__(cls, 'vertices')  # Manual late binding.


class Pentagon(FixedSizePolygon, vertex_count=5):
    pass


class Hexagon(FixedSizePolygon, vertex_count=6):
    pass


assert Pentagon.vertices.storage_name == '_vertices'
assert (Pentagon.vertex_count, Hexagon.vertex_count) == (5, 6)
assert Pentagon.vertices is not Hexagon.vertices
print('New descriptors:', Pentagon.vertices.min_length, Hexagon.vertices.max_length)

New descriptors: 5 6


### Step 2 — Verify class construction and inherited methods

Construct a valid five-point instance, then try to append. No Pentagon-specific append implementation exists. The inherited method must respect the descriptor installed by `__init_subclass__`.

In [31]:
five = Pentagon(Point2D(0, 0), Point2D(4, 0), Point2D(5, 2),
                Point2D(2, 5), Point2D(0, 3))
assert len(five) == 5
before = five._vertices
expect_error(ValueError, lambda: five.append(Point2D(1, 1)), 'at most')
assert five._vertices == before
expect_error(ValueError, lambda: type('BadCount', (FixedSizePolygon,), {}, vertex_count=True),
             'exact integer')
print('Fixed-size factory and constraint checks succeeded')

Fixed-size factory and constraint checks succeeded


**Design boundary:** this pattern is helpful when the sole per-subclass variation is count. If subclasses also need complicated shape-specific rules, keep those rules in their `_validate_vertices` methods. Remember that `vertex_count=5` says nothing about regularity: our `Pentagon` is **not** guaranteed to be a regular pentagon.

**Checkpoint.** What breaks if the explicit `descriptor.__set_name__(cls, "vertices")` line is removed?

**Solution / reasoning.** The late-assigned descriptor never learns its public and private names. Its `__set__` would then try to read missing descriptor metadata. Descriptors assigned in a class body do not have this issue.

## Problem 10 — Normalize polygon winding without changing its shape

**The problem.** A simple polygon may be stored clockwise or counterclockwise. Algorithms sometimes assume one orientation.

**Your task.** Use the signed double area to identify winding, create a counterclockwise **copy** while preserving the first vertex, and show that reversing orientation changes the sign but not the geometric region.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Relate signed area to traversal direction

The shoelace sum `Σ(x_i y_{i+1} − x_{i+1} y_i)` is twice the signed area. In the conventional mathematical coordinate plane, positive means counterclockwise. Even if a screen displays y downward, the algebraic sign is well defined as long as the convention is stated.

In [32]:
def winding(poly):
    twice = signed_twice_area(poly._vertices)
    if twice == 0:
        return 'degenerate'
    return 'counterclockwise' if twice > 0 else 'clockwise'


forward = simple_of((0, 0), (4, 0), (4, 4), (0, 4))
backward = SimplePolygon(*reversed(forward.vertices))
assert winding(forward) == 'counterclockwise'
assert winding(backward) == 'clockwise'
assert abs(signed_twice_area(forward._vertices)) == abs(signed_twice_area(backward._vertices))
print(winding(forward), '|', winding(backward))

counterclockwise | clockwise


### Step 2 — Make a copy, do not mutate the source

We keep the first point fixed and reverse the remaining order. This produces the reversed traversal starting from the same vertex. Calling `type(poly)` preserves the class and therefore its descriptor and validation rules; failure to reconstruct would surface as an ordinary validation error.

In [33]:
def counterclockwise_copy(poly):
    points = poly.vertices
    if signed_twice_area(poly._vertices) > 0:
        return type(poly)(*points)
    reordered = (points[0], *reversed(points[1:]))
    return type(poly)(*reordered)


normalized = counterclockwise_copy(backward)
assert winding(normalized) == 'counterclockwise'
assert normalized[0] == backward[0]
assert backward._vertices != normalized._vertices
assert winding(backward) == 'clockwise'  # Original unchanged.
assert type(normalized) is type(backward)
print('Normalized copy:', normalized)

Normalized copy: SimplePolygon(Point2D(0, 4), Point2D(0, 0), Point2D(4, 0), Point2D(4, 4))


**Not a square test, not a convexity test:** winding tells us the orientation of the ordered boundary and signed area. It does not by itself establish simplicity or convexity; that is why this operation is illustrated with `SimplePolygon`.

**Checkpoint.** Why preserve the first vertex when reversing?

**Solution / reasoning.** It is not geometrically necessary, but it keeps a useful indexing convention stable, simplifies tests, and avoids surprising callers who treat vertex zero as a named starting corner.

## Problem 11 — Classify a query point: inside, outside, or boundary

**The problem.** An application needs to know whether a pixel location lies inside a simple polygon. A boolean answer loses an important distinction: a point may lie exactly on the boundary.

**Your task.** Implement a three-way classifier using an exact boundary test followed by a half-open ray-crossing rule. Test concave shapes and reversal invariance.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Decide the input and output contracts first

The classifier takes a `SimplePolygon` and a `Point2D` and returns one of the strings `"inside"`, `"outside"`, or `"boundary"`. Boundary classification happens **first**, because the parity rule alone is delicate for points on edges or vertices. All tests below operate on integer pairs, so comparisons can avoid floating-point division.

### Step 2 — Why use a half-open crossing rule?

Cast a horizontal ray to the right from the query point. Count boundary edges that cross the ray: an odd count indicates inside, an even count indicates outside. If we count both ends of an edge at a vertex, we could double-count. Use `a_y <= p_y < b_y` for upward edges, or the corresponding downward condition, so each vertex is counted consistently. The horizontal edge case is already handled by the boundary check and otherwise contributes no crossing.

In [34]:
def classify_point(poly, point):
    if not isinstance(poly, SimplePolygon):
        raise TypeError('poly must be a SimplePolygon')
    if not isinstance(point, Point2D):
        raise TypeError('point must be a Point2D')

    q = (point.x, point.y)
    coords = poly._vertices
    inside = False
    for i, a in enumerate(coords):
        b = coords[(i + 1) % len(coords)]
        if point_on_segment(a, b, q):
            return 'boundary'

        ay, by, py = a[1], b[1], q[1]
        if (ay <= py < by) or (by <= py < ay):
            # Is the x-coordinate of the intersection strictly RIGHT of q?
            denominator = by - ay
            lhs = (q[0] - a[0]) * denominator
            rhs = (py - ay) * (b[0] - a[0])
            crosses_right = lhs < rhs if denominator > 0 else lhs > rhs
            if crosses_right:
                inside = not inside
    return 'inside' if inside else 'outside'


unit_box = simple_of((0, 0), (6, 0), (6, 6), (0, 6))
assert classify_point(unit_box, Point2D(3, 3)) == 'inside'
assert classify_point(unit_box, Point2D(7, 3)) == 'outside'
assert classify_point(unit_box, Point2D(3, 0)) == 'boundary'
assert classify_point(unit_box, Point2D(0, 0)) == 'boundary'
print('Interior, exterior, edge, and vertex checks passed')

Interior, exterior, edge, and vertex checks passed


### Step 3 — Explain the cross-multiplication

For an active edge, the ray hits it at `x_hit = a_x + (p_y−a_y)(b_x−a_x)/(b_y−a_y)`. We need `x_hit > p_x`. Cross-multiplication removes division, but when `(b_y−a_y)` is negative the inequality reverses. The two comparison branches in the code are intentional, not an optimization gimmick.

### Step 4 — Use a concave example that a bounding box cannot solve

The polygon below has a V-shaped indentation near its top. The point `(3, 5)` is inside its bounding box yet outside the polygon. This is why a bounding-box-only test is insufficient.

In [35]:
notched = simple_of((0, 0), (6, 0), (6, 6), (3, 3), (0, 6))
checks = { (3, 1): 'inside', (3, 5): 'outside',
           (3, 3): 'boundary', (5, 3): 'inside', (7, 1): 'outside' }
for xy, expected in checks.items():
    actual = classify_point(notched, Point2D(*xy))
    assert actual == expected, (xy, expected, actual)
print('Concave-polygon cases:', checks)

Concave-polygon cases: {(3, 1): 'inside', (3, 5): 'outside', (3, 3): 'boundary', (5, 3): 'inside', (7, 1): 'outside'}


### Step 5 — Reversing vertex order must not change classification

The boundary encloses the same region regardless of traversal direction. Reverse the polygon and test the same queries again. This is a metamorphic test: it verifies a property even without a separate reference implementation.

In [36]:
notched_reversed = SimplePolygon(*reversed(notched.vertices))
for xy, expected in checks.items():
    assert classify_point(notched_reversed, Point2D(*xy)) == expected
expect_error(TypeError, lambda: classify_point(poly, Point2D(1, 1)), 'SimplePolygon')
expect_error(TypeError, lambda: classify_point(notched, (1, 1)), 'Point2D')
print('Winding-independent classification and input contracts verified')

Winding-independent classification and input contracts verified


**Checkpoint.** Why do we not use `float` intersection coordinates?

**Solution / reasoning.** Integer cross-multiplication is exact for these integer inputs and avoids rounding near edges. The boundary predicate resolves exact edge membership before the parity calculation.

## Problem 12 — Translate and quarter-turn polygons with safe rollback

**The problem.** Transforms can push coordinates beyond the permitted screen rectangle, and an in-place implementation may update the first vertex before discovering that the last vertex is invalid.

**Your task.** Create both copy-returning and in-place transformations using a single candidate assignment, then implement an exact 90-degree rotation about an integer center.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Validate transform parameters rather than coercing them

Translations preserve integer coordinates if and only if the offsets are integers. We reject boolean offsets and floating-point values rather than silently truncating them. New `Point2D` objects validate the bounds; class construction or descriptor assignment validates the resulting polygon as a whole.

In [37]:
def exact_int(value, name):
    if type(value) is not int:
        raise TypeError(f'{name} must be an exact int')
    return value


def translated_copy(poly, dx, dy):
    exact_int(dx, 'dx')
    exact_int(dy, 'dy')
    moved = tuple(Point2D(p.x + dx, p.y + dy) for p in poly.vertices)
    return type(poly)(*moved)


moving = simple_of((1, 1), (3, 1), (3, 3), (1, 3))
shifted = translated_copy(moving, 5, 7)
assert shifted._vertices == ((6, 8), (8, 8), (8, 10), (6, 10))
assert moving._vertices == ((1, 1), (3, 1), (3, 3), (1, 3))
assert signed_twice_area(shifted._vertices) == signed_twice_area(moving._vertices)
print('Copy transform preserves source and signed area')

Copy transform preserves source and signed area


### Step 2 — In-place translation must assign only once

Build the entire candidate before writing `poly.vertices`. If a coordinate is out of range, point construction raises before any commit. If the shape validator rejects the result for another reason, the descriptor likewise rejects it before storage.

In [38]:
def translate_inplace(poly, dx, dy):
    exact_int(dx, 'dx')
    exact_int(dy, 'dy')
    moved = tuple(Point2D(p.x + dx, p.y + dy) for p in poly.vertices)
    poly.vertices = moved
    return poly


near_edge = simple_of((798, 1), (800, 1), (800, 3), (798, 3))
before = near_edge._vertices
expect_error(ValueError, lambda: translate_inplace(near_edge, 1, 0), '<= 800')
assert near_edge._vertices == before
assert translate_inplace(near_edge, -1, 0) is near_edge
assert near_edge[0] == Point2D(797, 1)
expect_error(TypeError, lambda: translated_copy(moving, True, 0), 'dx')
print('Out-of-bounds rollback and valid in-place translation passed')

Out-of-bounds rollback and valid in-place translation passed


### Step 3 — Rotate with integer arithmetic only

About center `(cx, cy)`, a counterclockwise quarter-turn sends `(x−cx, y−cy)` to `(-(y−cy), x−cx)`. Repeat the transformation `quarter_turns % 4` times. The input center and turn count must be integers. Constructing a new instance preserves its class-specific constraints.

In [39]:
def quarter_turn_copy(poly, cx, cy, quarter_turns=1):
    for name, value in (('cx', cx), ('cy', cy), ('quarter_turns', quarter_turns)):
        exact_int(value, name)
    new_points = []
    for p in poly.vertices:
        x, y = p.x - cx, p.y - cy
        for _ in range(quarter_turns % 4):
            x, y = -y, x
        new_points.append(Point2D(cx + x, cy + y))
    return type(poly)(*new_points)


rotated = quarter_turn_copy(moving, 2, 2, 1)
assert rotated._vertices == ((3, 1), (3, 3), (1, 3), (1, 1))
assert quarter_turn_copy(moving, 2, 2, 4)._vertices == moving._vertices
assert quarter_turn_copy(moving, 2, 2, -1)._vertices == quarter_turn_copy(moving, 2, 2, 3)._vertices
assert winding(rotated) == winding(moving)
print('Quarter-turn rotation and identity/inverse relations checked')

Quarter-turn rotation and identity/inverse relations checked


**Limitation to remember:** a quarter-turn about a center near a screen edge may produce invalid coordinates. In that case this function raises; it does not clip the polygon because clipping can alter geometry and vertex count. A general rotation by an arbitrary angle would need a separately specified rounding policy, which we intentionally avoid here.

**Checkpoint.** Can `translate_inplace` leave two transformed vertices and two original vertices if the last point is out of range?

**Solution / reasoning.** No. It builds all new points first and performs exactly one validated assignment afterward. Exceptions during point construction prevent the assignment from happening at all.

## Problem 13 — When `WeakKeyDictionary` is the wrong descriptor storage

**The problem.** A common descriptor recipe stores per-instance values in `weakref.WeakKeyDictionary`. That can be useful, but does it work for every point class?

**Your task.** Demonstrate two practical hazards—unhashable keys and equality-based key collisions—then connect the findings to our private-slot design.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — A mutable value-equal point is deliberately unhashable

`WeakKeyDictionary` needs hashable, weak-referenceable keys. Our `Point2D` is unhashable because its coordinates can change. It also has slots without `__weakref__`. The attempted weak map insertion fails; a weak map is not a universal escape hatch from `__dict__`.

In [40]:
weak_map = weakref.WeakKeyDictionary()
expect_error(TypeError, lambda: weak_map.__setitem__(Point2D(1, 2), 'data'))
expect_error(TypeError, lambda: weakref.ref(Point2D(1, 2)))
print('Mutable slotted points are neither usable weak keys nor weak-referenceable')

Mutable slotted points are neither usable weak keys nor weak-referenceable


### Step 2 — Even hashable value-equal keys can surprise you

Now use a separate demonstration class that is weak-referenceable and hashable **by value**. Two distinct objects compare equal, so a dictionary treats them as the same key. A weak-key mapping is still a mapping with equality/hash semantics: it is not automatically an identity map.

In [41]:
class ValueKey:
    def __init__(self, number):
        self.number = number

    def __eq__(self, other):
        return isinstance(other, ValueKey) and self.number == other.number

    def __hash__(self):
        return hash(self.number)


a, b = ValueKey(5), ValueKey(5)
assert a is not b and a == b
mapping = weakref.WeakKeyDictionary()
mapping[a] = 'first'
mapping[b] = 'second'
assert len(mapping) == 1
assert mapping[a] == mapping[b] == 'second'
print('Two distinct but equal objects occupy one logical mapping key')

Two distinct but equal objects occupy one logical mapping key


### Step 3 — State the engineering conclusion precisely

Weak-key storage is appropriate when the descriptor author controls key requirements and wants its values released with the instance. For our user-facing mutable, value-equal, slotted points, private slots are more straightforward and avoid weak-map equality surprises. No single storage strategy is universally best.

**Checkpoint.** Would adding `__weakref__` to `Point2D.__slots__` solve both issues?

**Solution / reasoning.** No. It would enable weak references, but the class would still be unhashable due to mutable value equality. Making it value-hashable would introduce a different correctness bug when coordinates change.

## Problem 14 — Capstone: combine editable geometry and test invariants repeatedly

**The problem.** Now we have validated geometry, editing methods, copy-safe transforms, and query algorithms. Can they work together without bypassing one another?

**Your task.** Combine mixins through multiple inheritance, test method resolution, test failures and successful edits, then run reproducible randomized checks against core invariants.

Try outlining a solution before revealing the code in the next cells. Every solution is divided into short, testable steps.

### Step 1 — Combine classes without copying implementations

`SimplePolygon` supplies the geometric hook, and `EditingPolygon` supplies explicit edit methods. Both derive from `Polygon`. Put the geometric class first and check the method-resolution order: the validator should come from `SimplePolygon`, while `append` should come from `EditablePolygon`.

In [42]:
class EditableSimplePolygon(SimplePolygon, EditingPolygon):
    """Simple boundary + safe append, extend, replacement, and deletion."""


class EditableConvexPolygon(StrictConvexPolygon, EditingPolygon):
    """Strict convexity + the same editing API."""


print('EditableSimplePolygon MRO:', [cls.__name__ for cls in EditableSimplePolygon.__mro__])
assert EditableSimplePolygon._validate_vertices is SimplePolygon._validate_vertices
assert EditableSimplePolygon.append is EditablePolygon.append
assert EditableConvexPolygon._validate_vertices is StrictConvexPolygon._validate_vertices

EditableSimplePolygon MRO: ['EditableSimplePolygon', 'SimplePolygon', 'EditingPolygon', 'EditablePolygon', 'Polygon', 'Sequence', 'Reversible', 'Collection', 'Sized', 'Iterable', 'Container', 'object']


### Step 2 — Test geometry validation through an inherited method

Start with a square and append a point that would create an invalid boundary. Because `append` performs a descriptor assignment, the subclass hook rejects the candidate and keeps the square unchanged.

In [43]:
capstone = EditableSimplePolygon(Point2D(0, 0), Point2D(4, 0),
                                 Point2D(4, 4), Point2D(0, 4))
old = capstone._vertices
expect_error(ValueError, lambda: capstone.append(Point2D(4, 2)), 'intersect')
assert capstone._vertices == old
capstone.append(Point2D(0, 2))  # Collinear forward subdivision of closing edge.
assert len(capstone) == 5
assert signed_twice_area(capstone._vertices) == signed_twice_area(old)
print('Inherited append respects geometry and accepts a valid subdivision')

Inherited append respects geometry and accepts a valid subdivision


### Step 3 — Strict convexity should reject the same collinear subdivision

A simple polygon may contain redundant, forward collinear vertices, while a **strictly** convex polygon may not. The exact same inherited method has different outcomes because the polymorphic hook enforces different contracts.

In [44]:
convex_capstone = EditableConvexPolygon(Point2D(0, 0), Point2D(4, 0),
                                             Point2D(4, 4), Point2D(0, 4))
old = convex_capstone._vertices
expect_error(ValueError, lambda: convex_capstone.append(Point2D(0, 2)), 'collinear')
assert convex_capstone._vertices == old
print('Identical edit API, stronger geometry contract')

Identical edit API, stronger geometry contract


### Step 4 — Build small randomized tests with a fixed seed

Rather than depend on external packages, generate 100 triangles on a 21 × 21 integer grid. Skip collinear triples. Every nondegenerate triangle is simple and strictly convex, so these are valid inputs. Translate by a small offset only when all moved coordinates remain on the screen; test area invariance and copy independence.

In [45]:
rng = random.Random(20260922)
valid_cases = 0
for _ in range(100):
    a, b, c = rng.sample([(x, y) for x in range(21) for y in range(21)], 3)
    if turn(a, b, c) == 0:
        continue
    shape = GeometricTriangle(Point2D(*a), Point2D(*b), Point2D(*c))
    original = shape._vertices
    dx, dy = rng.randrange(0, 6), rng.randrange(0, 6)
    moved = translated_copy(shape, dx, dy)
    assert isinstance(moved, GeometricTriangle)
    assert shape._vertices == original
    assert signed_twice_area(moved._vertices) == signed_twice_area(original)
    assert winding(counterclockwise_copy(shape)) == 'counterclockwise'
    valid_cases += 1
assert valid_cases >= 90
print('Random nondegenerate triangle cases passed:', valid_cases)

Random nondegenerate triangle cases passed: 99


### Step 5 — Run a randomized rollback test

For every random polygon below, invalidly assign a two-point list and ensure the private state stays identical. This seems elementary, but it tests the *same invariant across many independently generated objects* and guards against accidental future refactoring that writes before validating.

In [46]:
for _ in range(60):
    left, bottom = rng.randrange(0, 780), rng.randrange(0, 380)
    width, height = rng.randrange(1, 11), rng.randrange(1, 11)
    shape = EditableConvexPolygon(Point2D(left, bottom), Point2D(left + width, bottom),
                                  Point2D(left + width, bottom + height), Point2D(left, bottom + height))
    saved = shape._vertices
    expect_error(ValueError, lambda: setattr(shape, 'vertices', [Point2D(0, 0), Point2D(1, 1)]), 'at least')
    assert shape._vertices == saved
    assert all(turn(saved[i-1], saved[i], saved[(i+1) % 4]) > 0 for i in range(4))
print('60 generated convex polygons survived invalid reassignment unchanged')

60 generated convex polygons survived invalid reassignment unchanged


### Capstone solution review

We developed a layered contract, not a monolithic polygon class:

- `StrictBoundedInt` validates an individual coordinate before storing it in a private slot.
- `VertexSequence` checks an entire sequence, snapshots coordinate values, invokes one extensible validation hook, and commits exactly once.
- `Polygon` exposes the familiar read-only `Sequence` interface without exposing internal mutable references.
- `EditingPolygon` defines edits in terms of complete replacement, so derived constraints are automatically rechecked.
- `SimplePolygon` and `StrictConvexPolygon` add different, explicitly named geometric guarantees.
- `FixedSizePolygon` demonstrates a class-creation extension point and the manually required `__set_name__` call.
- Query and transformation functions keep their input/output contracts explicit and never infer “square” from vertex count.

**Remaining limits:** `__slots__` and descriptors are not a security boundary; code can deliberately write private slots. The implementation assumes one thread edits a polygon at a time. Boundary testing costs O(n²) in this educational version. A full production geometry library would additionally specify numeric overflow limits, threading behavior, polygon holes, and complex boundary topologies.

## Further independent exercises (prompts + solution outlines)

These are optional extensions; the tested solutions above are fully runnable without completing them.

**Exercise A — Duplicate points in non-simple polygons.** Prompt: should an unrestricted `Polygon` reject repeated consecutive points? **Solution outline:** decide whether it represents arbitrary vertex data or geometric boundaries. If it promises only a typed sequence, keep the base permissive; apply distinctness in `SimplePolygon`, as done above.

**Exercise B — Avoid O(n) point reconstruction per index.** Prompt: `poly[0]` currently copies every point. **Solution outline:** for integer indices, read only `self._vertices[key]` and return `Point2D(*pair)`; for slices, convert only the sliced pairs. Preserve detached snapshots. Benchmark before treating this as a necessary optimization.

**Exercise C — Lock an entire read-modify-write edit.** Prompt: make two concurrent `append` calls unable to lose an update. **Solution outline:** initialize one per-instance reentrant lock, acquire it around candidate construction *and* commit, and define a locking policy for direct descriptor assignments. Locking only `__set__` is insufficient because two threads can construct candidates from the same old snapshot.

**Exercise D — Test a geometric oracle.** Prompt: generate many integer rectangles and enumerate all integer grid points; compare the classifier against the simple closed-form rectangle rule, including boundaries. **Solution outline:** classify `x = left/right` or `y = bottom/top` (when the other coordinate is in range) as boundary, strict interior as inside, and all remaining points as outside. Test both traversal directions.

**Exercise E — Perimeter with exact squared distances.** Prompt: compare polygons by squared edge lengths without introducing floating point until final measurement. **Solution outline:** compute `(dx*dx + dy*dy)` for each edge; use `math.sqrt` only to report physical perimeter. State that equal squared edge lengths alone do not establish that a quadrilateral is a square.

## Final executable review — essential promises

A short final cell checks the public behavior without relying on printed output or notebook-specific state displays. If it prints the success message, the central descriptor, snapshot, sequence, subclass, geometry, transform, and classifier contracts all hold for these representative cases.

In [47]:
final_point = Point2D(2, 3)
assert Point2D.x.storage_name == '_x'
expect_error(TypeError, lambda: setattr(final_point, 'x', False))
final_shape = EditableSimplePolygon(Point2D(0, 0), Point2D(6, 0),
                                    Point2D(6, 6), Point2D(0, 6))
saved = final_shape._vertices
view = final_shape.vertices
view[0].x = 100
assert final_shape._vertices == saved
assert isinstance(final_shape, Sequence)
assert classify_point(final_shape, Point2D(2, 2)) == 'inside'
assert classify_point(final_shape, Point2D(2, 0)) == 'boundary'
assert classify_point(final_shape, Point2D(7, 2)) == 'outside'
expect_error(ValueError, lambda: final_shape.append(Point2D(6, 3)))
assert final_shape._vertices == saved
assert translated_copy(final_shape, 2, 3)._vertices[0] == (2, 3)
assert final_shape._vertices == saved
print('SUCCESS: core contracts and cross-problem integration passed')

SUCCESS: core contracts and cross-problem integration passed
